In [0]:
source_table = "test.test_schema.customer_master"
target_table = "test.test_schema.customer_profile"
checkpoint_table = "test.test_schema.cdf_pipeline_checkpoint"
pipeline_name = "customer_master_to_customer_profile"
your_table = "test.test_schema.customer_master"

# Option A: Truncate (keeps schema, removes all rows)
#spark.sql(f"TRUNCATE TABLE {source_table}")

# Option B: Drop and recreate
#spark.sql(f"DROP TABLE IF EXISTS {source_table}")
# Then recreate via your usual write logic

# Option A: Truncate (keeps schema, removes all rows)
#park.sql(f"TRUNCATE TABLE {target_table}")

# Option B: Drop and recreate
#spark.sql(f"DROP TABLE IF EXISTS {target_table}")
# Then recreate via your usual write logic

In [0]:
%sql
--select * from test.test_schema.customer_master version as of 18;

In [0]:
%sql
--restore test.test_schema.customer_master version as of 18;

In [0]:
%sql
--describe history test.test_schema.customer_master;

In [0]:
%sql
select * from test.test_schema.customer_profile;

In [0]:
%sql
--delete from test.test_schema.customer_master;
--delete from  test.test_schema.customer_profile;
--delete from test.test_schema.cdf_pipeline_checkpoint


In [0]:
%sql
create catalog if not exists test;
create schema if not exists test.test_schema; 

In [0]:
%sql
CREATE TABLE IF NOT EXISTS test.test_schema.customer_master (
 customer_id BIGINT,
 customer_name STRING,
 email STRING,
 city STRING,
 customer_status STRING,
 updated_at TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
 'delta.enableChangeDataFeed' = 'true'
);


In [0]:
%sql
SHOW TBLPROPERTIES test.test_schema.customer_master (
 'delta.enableChangeDataFeed'
);


In [0]:
%sql
CREATE TABLE IF NOT EXISTS test.test_schema.customer_profile (
 customer_id BIGINT,
 customer_name STRING,
 email STRING,
 city STRING,
 customer_status STRING,
 updated_at TIMESTAMP,
 source_commit_version BIGINT,
 source_commit_timestamp TIMESTAMP

)
USING DELTA;


In [0]:
%sql
INSERT INTO test.test_schema.customer_master VALUES
(101, 'Ananya Sharma', 'ananya.sharma@example.com', 'Bengaluru', 'ACTIVE', current_timestamp()),
(102, 'Rahul Verma', 'rahul.verma@example.com', 'Mumbai', 'ACTIVE', current_timestamp()),
(103, 'Priya Nair', 'priya.nair@example.com', 'Chennai', 'ACTIVE', current_timestamp());


In [0]:
%sql
describe history test.test_schema.customer_master;

In [0]:
from pyspark.sql import functions as F
source_table = "test.test_schema.customer_master"
target_table = "test.test_schema.customer_profile"
checkpoint_table = "test.test_schema.cdf_pipeline_checkpoint"
pipeline_name = "customer_master_to_customer_profile"

def apply_customer_cdf(starting_version, ending_version):
 cdf_df = (
 spark.read
 .option("readChangeFeed", "true")
 .option("startingVersion", starting_version)
 .option("endingVersion", ending_version)
 .table(source_table)
 )

 changes_df = (
 cdf_df
 .filter(F.col("_change_type").isin("insert", "update_postimage", "delete"))
 .select(
 "customer_id", "customer_name", "email", "city",
 "customer_status", "updated_at", "_change_type",
 "_commit_version", "_commit_timestamp"
  )
 )

 changes_df.createOrReplaceTempView("customer_cdf_changes")

 spark.sql(f"""
 MERGE INTO {target_table} AS target
 USING customer_cdf_changes AS source
 ON target.customer_id = source.customer_id
 WHEN MATCHED AND source._change_type = 'delete'
 THEN DELETE
 WHEN MATCHED
 AND source._change_type IN ('insert', 'update_postimage')
 THEN UPDATE SET
 target.customer_name = source.customer_name,
 target.email = source.email,
 target.city = source.city,
 target.customer_status = source.customer_status,
 target.updated_at = source.updated_at,
 target.source_commit_version = source._commit_version,
 target.source_commit_timestamp = source._commit_timestamp

 WHEN NOT MATCHED
 AND source._change_type IN ('insert', 'update_postimage')
 THEN INSERT (
 customer_id, customer_name, email, city, customer_status,
 updated_at, source_commit_version, source_commit_timestamp
 ) VALUES (
 source.customer_id, source.customer_name, source.email,
 source.city, source.customer_status, source.updated_at,
 source._commit_version, source._commit_timestamp
 )
 """)
 
 return changes_df

In [0]:
day1_ending_version = (
 spark.sql(f"DESCRIBE HISTORY {source_table}")
  .select(F.max("version").alias("latest_version"))
 .first()["latest_version"]
)
day1_changes = apply_customer_cdf(
 starting_version=0,
 ending_version=day1_ending_version
)
display(day1_changes.orderBy("_commit_version", "customer_id"))

In [0]:
spark.sql(f"""
 MERGE INTO {checkpoint_table} AS target
 USING (
 SELECT
 '{pipeline_name}' AS pipeline_name,
 CAST({day1_ending_version} AS BIGINT) AS last_processed_version,
 current_timestamp() AS updated_at
 ) AS source
 ON target.pipeline_name = source.pipeline_name
 WHEN MATCHED THEN UPDATE SET
 target.last_processed_version = source.last_processed_version,
 target.updated_at = source.updated_at
 
 WHEN NOT MATCHED THEN INSERT (
 pipeline_name, last_processed_version, updated_at
 ) VALUES (
 source.pipeline_name, source.last_processed_version, source.updated_at
 )
""")


In [0]:
display(spark.table(target_table).orderBy("customer_id"))
display(spark.table(checkpoint_table))

In [0]:
%sql
SELECT *
FROM test.test_schema.customer_profile
ORDER BY customer_id;

In [0]:
%sql
SELECT
 customer_id,
 customer_name,
 city,
 customer_status,
 source_commit_version
FROM test.test_schema.customer_profile
ORDER BY customer_id;
